# Africa Fuel Tracker — Initial State Builder
## `01_initial_state.ipynb`

This notebook:
1. Loads the corrected reference data (54 countries, validated values)
2. Builds `data/prices_db.json` — the **current state** file
3. Builds `data/history_db.json` — the **historical** file (Jan 2026 → today)
4. For historical data, fetches available archives from BCEAO bulletins + aggregators
5. Validates all values and generates a summary report

**Run once** before activating the daily GitHub Actions pipeline.


### 1. Install dependencies

In [ ]:
%pip install -q requests beautifulsoup4 lxml nbformat
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # project root
print("✅ Dependencies ready")


### 2. Imports & configuration

In [ ]:
import json
import requests
import re
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path
from bs4 import BeautifulSoup

# Project root (notebook is in notebooks/)
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

PRICES_DB  = DATA_DIR / 'prices_db.json'
HISTORY_DB = DATA_DIR / 'history_db.json'

# Date range: 1-Jan-2026 → today
START_DATE = date(2026, 1, 1)
TODAY      = date.today()

print(f"Project root : {ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Date range   : {START_DATE} → {TODAY}")


### 3. Reference data (54 countries — validated values)

In [ ]:
# ─── VALIDATED REFERENCE DATA ─────────────────────────────────────────────────
# Source: official_sources_results_54.json + corrections applied
# All values verified against official regulators (see africa_fuel_sources_officielles_54pays.xlsx)

REFERENCE_DATA = {
    # country: (iso2, region, currency, gas_loc, die_loc, fx_rate, source_url, eff_date, confidence, update_cycle)
    "Algeria":          ("DZ","North Africa",  "DZD",  47.0,    31.0,    532.0,  "https://www.thefuelprice.com/Fdz/en",               "2026-01-01","high",  "stable"),
    "Egypt":            ("EG","North Africa",  "EGP",  24.0,    20.5,    50.3,   "https://www.sis.gov.eg/en/media-center",             "2026-03-10","high",  "quarterly"),
    "Libya":            ("LY","North Africa",  "LYD",  0.15,    0.135,   4.85,   "https://noc.ly/",                                   "2026-01-01","high",  "stable"),
    "Morocco":          ("MA","North Africa",  "MAD",  13.82,   12.76,   10.03,  "https://auto24.ma/en/fuel-price-tracker",            "2026-03-16","high",  "weekly"),
    "Sudan":            ("SD","North Africa",  "SDG",  630.0,   590.0,   601.5,  "https://oilpricez.com/sd/sudan-gasoline-price",      "2026-03-29","low",   "variable"),
    "Tunisia":          ("TN","North Africa",  "TND",  2.53,    2.21,    3.02,   "https://www.stir.com.tn/",                          "2026-01-01","high",  "stable"),
    "Benin":            ("BJ","West Africa",   "XOF",  740.0,   760.0,   603.5,  "https://www.gouv.bj/",                              "2026-02-01","medium","monthly"),
    "Burkina Faso":     ("BF","West Africa",   "XOF",  840.0,   724.0,   603.5,  "https://www.sonabhy.bf/",                           "2026-02-01","medium","monthly"),
    "Cabo Verde":       ("CV","West Africa",   "CVE",  129.5,   108.8,   100.5,  "https://www.are.cv/",                               "2026-03-01","medium","monthly"),
    "Gambia":           ("GM","West Africa",   "GMD",  74.0,    65.0,    68.0,   "https://www.gambia.gov.gm/",                        "2026-01-01","medium","quarterly"),
    "Ghana":            ("GH","West Africa",   "GHS",  14.35,   14.35,   10.89,  "https://www.npa.gov.gh/",                           "2026-03-16","high",  "bimonthly"),
    "Guinea":           ("GN","West Africa",   "GNF",  11834.0, 11600.0, 8650.0, "https://www.mines.gov.gn/",                         "2026-02-01","medium","monthly"),
    "Guinea-Bissau":    ("GW","West Africa",   "XOF",  802.0,   766.0,   603.5,  "https://www.bceao.int/fr/publications/bulletins",   "2025-06-30","medium","variable"),
    "Ivory Coast":      ("CI","West Africa",   "XOF",  820.0,   675.0,   603.5,  "https://www.petroci.ci/",                           "2026-02-01","medium","monthly"),
    "Liberia":          ("LR","West Africa",   "LRD",  167.0,   177.0,   194.0,  "https://www.nocal.com.lr/",                         "2026-02-01","medium","quarterly"),
    "Mali":             ("ML","West Africa",   "XOF",  840.0,   754.0,   603.5,  "https://www.energie.gov.ml/",                       "2026-02-01","medium","monthly"),
    "Mauritania":       ("MR","West Africa",   "MRU",  42.6,    40.2,    36.5,   "https://www.petrole.gov.mr/",                       "2026-01-01","medium","quarterly"),
    "Niger":            ("NE","West Africa",   "XOF",  541.0,   605.0,   603.5,  "https://www.sonidep.ne/",                           "2026-02-01","medium","monthly"),
    "Nigeria":          ("NG","West Africa",   "NGN",  1130.0,  1600.0,  1621.0, "https://www.nnpcgroup.com/",                        "2026-03-18","medium","weekly"),
    "Senegal":          ("SN","West Africa",   "XOF",  920.0,   680.0,   603.5,  "https://primature.sn/publications/communiques-officiels/baisse-des-prix-des-hydrocarbures", "2025-12-06","high","variable"),
    "Sierra Leone":     ("SL","West Africa",   "SLL",  32000.0, 35000.0, 22500.0,"https://www.npra.gov.sl/",                          "2026-03-14","high",  "monthly"),
    "Togo":             ("TG","West Africa",   "XOF",  737.0,   713.0,   603.5,  "https://www.mines-energie.gouv.tg/",                "2026-02-01","medium","monthly"),
    "CAR":              ("CF","Central Africa","XAF",  1050.0,  1300.0,  603.5,  "https://www.primature.cf/",                         "2026-01-05","medium","quarterly"),
    "Cameroon":         ("CM","Central Africa","XAF",  840.0,   828.0,   603.5,  "https://www.csph.cm/",                              "2026-03-02","medium","monthly"),
    "Chad":             ("TD","Central Africa","XAF",  730.0,   828.0,   603.5,  "https://www.finances.gouv.td/",                     "2024-02-01","medium","stable"),
    "Congo DR":         ("CD","Central Africa","CDF",  2968.0,  2431.0,  2820.0, "https://www.hydrocarbures.gouv.cd/",                "2026-02-01","medium","monthly"),
    "Congo Republic":   ("CG","Central Africa","XAF",  695.0,   630.0,   603.5,  "https://www.hydrocarbures.gouv.cg/",                "2026-01-01","medium","quarterly"),
    "Equatorial Guinea":("GQ","Central Africa","XAF",  550.0,   520.0,   603.5,  "https://www.globalpetrolprices.com/Equatorial-Guinea/","2025-01-01","medium","quarterly"),
    "Gabon":            ("GA","Central Africa","XAF",  644.0,   576.0,   603.5,  "https://www.hydrocarbures.gouv.ga/",                "2026-01-01","medium","quarterly"),
    "Sao Tome":         ("ST","Central Africa","STN",  28.0,    25.5,    21.5,   "https://www.globalpetrolprices.com/Sao-Tome-and-Principe/","2025-06-01","medium","quarterly"),
    "Burundi":          ("BI","East Africa",   "BIF",  3936.0,  3761.0,  2920.0, "https://www.areen.bi/",                             "2026-02-01","medium","bimonthly"),
    "Comoros":          ("KM","East Africa",   "KMF",  760.0,   680.0,   426.0,  "https://oilpricez.com/km/comoros-oil-price",        "2026-03-16","medium","quarterly"),
    "Djibouti":         ("DJ","East Africa",   "DJF",  240.0,   225.0,   177.7,  "https://www.ane.gouv.dj/",                          "2026-01-01","medium","quarterly"),
    "Eritrea":          ("ER","East Africa",   "ERN",  30.0,    20.0,    15.0,   "https://www.thefuelprice.com/Fer/en",               "2016-12-01","low",   "stable"),
    "Ethiopia":         ("ET","East Africa",   "ETB",  132.18,  139.84,  157.4,  "https://www.2merkato.com/news/energy-and-mining/8814-ethiopia-ministry-of-trade-and-regional-integration-announces-fuel-price-adjustments","2026-03-10","high","monthly"),
    "Kenya":            ("KE","East Africa",   "KES",  178.28,  166.54,  130.5,  "https://www.epra.go.ke/pump-prices",                "2026-03-15","high",  "monthly"),
    "Madagascar":       ("MG","East Africa",   "MGA",  5516.0,  5174.0,  4620.0, "https://www.omh.mg/",                               "2026-03-01","medium","monthly"),
    "Malawi":           ("MW","East Africa",   "MWK",  4965.0,  4945.0,  1734.0, "https://www.meramalawi.mw/",                        "2026-02-01","medium","monthly"),
    "Mauritius":        ("MU","East Africa",   "MUR",  58.45,   64.80,   46.5,   "https://www.stcmu.com/ppm/press-release",           "2026-03-24","high",  "monthly"),
    "Rwanda":           ("RW","East Africa",   "RWF",  1989.0,  1948.0,  1463.0, "https://www.rura.rw/index.php/en/fuel-prices",      "2026-03-05","high",  "bimonthly"),
    "Seychelles":       ("SC","East Africa",   "SCR",  19.96,   18.82,   14.15,  "https://www.puc.sc/",                               "2026-02-01","medium","monthly"),
    "Somalia":          ("SO","East Africa",   "USD",  1.15,    1.05,    1.0,    "https://nbs.gov.so/",                               "2026-03-10","medium","variable"),
    "South Sudan":      ("SS","East Africa",   "SSP",  12000.0, 12000.0, 4544.0, "https://www.eyeradio.org/fuel-price-surge-a-wake-up-call-for-government-to-protect-citizens/","2026-03-06","medium","variable"),
    "Tanzania":         ("TZ","East Africa",   "TZS",  2864.0,  2858.0,  2710.0, "https://www.ewura.go.tz/faqs/petroleum-fuel-prices","2026-03-04","high",  "monthly"),
    "Uganda":           ("UG","East Africa",   "UGX",  4950.0,  4564.0,  3582.0, "https://www.pau.go.ug/",                            "2026-03-16","medium","weekly"),
    "Angola":           ("AO","Southern Africa","AOA", 300.0,   400.0,   917.0,  "https://www.ianpetro.ao/",                          "2025-12-01","medium","stable"),
    "Botswana":         ("BW","Southern Africa","BWP", 15.47,   16.28,   13.54,  "https://www.bera.co.bw/",                           "2026-03-09","medium","monthly"),
    "Eswatini":         ("SZ","Southern Africa","SZL", 19.89,   21.27,   18.55,  "https://www.dmre.gov.za/energy-resources/energy-sources/pretoleum/fuel-prices","2026-03-04","high","monthly"),
    "Lesotho":          ("LS","Southern Africa","LSL", 17.95,   19.40,   17.4,   "https://www.energy.gov.ls/",                        "2026-03-04","high",  "monthly"),
    "Mozambique":       ("MZ","Southern Africa","MZN", 83.97,   80.12,   64.1,   "https://www.inp.gov.mz/",                           "2026-03-01","medium","monthly"),
    "Namibia":          ("NA","Southern Africa","NAD", 19.58,   19.63,   18.55,  "https://www.mme.gov.na/news/148/Fuel-Price-Review-Announcement-March-2026","2026-03-03","high","monthly"),
    "South Africa":     ("ZA","Southern Africa","ZAR", 19.89,   21.27,   18.55,  "https://www.dmre.gov.za/energy-resources/energy-sources/pretoleum/fuel-prices","2026-03-04","high","monthly"),
    "Zambia":           ("ZM","Southern Africa","ZMW", 26.61,   23.25,   26.2,   "https://www.erb.org.zm/category/price-build-up",    "2026-02-28","high",  "monthly"),
    "Zimbabwe":         ("ZW","Southern Africa","ZWG", 2170.0,  2050.0,  26.0,   "https://www.zera.co.zw/",                           "2026-03-18","high",  "weekly"),
}

print(f"✅ Reference data loaded: {len(REFERENCE_DATA)} countries")
print(f"   By region:")
from collections import Counter
regions = Counter(v[1] for v in REFERENCE_DATA.values())
for region, count in sorted(regions.items(), key=lambda x: -x[1]):
    print(f"     {region:<20} {count} countries")


### 4. Fetch today's FX rates (Frankfurter API)

In [ ]:
def get_fx_rates():
    """Fetch all needed FX rates from Frankfurter API + fixed rates."""
    FIXED = {
        "LYD": 4.85, "SDG": 601.5, "SSP": 4544.0, "ERN": 15.0,
        "SLL": 22500.0, "STN": 21.5, "AOA": 917.0, "CDF": 2820.0,
        "ZWG": 26.0,
    }
    # Currencies to fetch from Frankfurter
    all_currencies = list(set(v[2] for v in REFERENCE_DATA.values()))
    to_fetch = [c for c in all_currencies if c not in FIXED and c != "USD"]
    
    rates = {"USD": 1.0}
    rates.update({k: v for k, v in FIXED.items()})
    
    if to_fetch:
        try:
            symbols = ",".join(sorted(set(to_fetch)))
            resp = requests.get(
                "https://api.frankfurter.app/latest",
                params={"from": "USD", "to": symbols},
                timeout=15
            )
            resp.raise_for_status()
            data = resp.json()
            rates.update(data.get("rates", {}))
            print(f"✅ Frankfurter: {len(data.get('rates',{}))} rates fetched (date: {data.get('date')})")
        except Exception as e:
            print(f"⚠️  Frankfurter failed: {e} — using reference FX rates")
            # Use reference FX rates from REFERENCE_DATA as fallback
            for country, vals in REFERENCE_DATA.items():
                cur = vals[2]; fx = vals[6]
                if cur not in rates and fx > 0:
                    rates[cur] = fx
    
    return rates

FX_RATES = get_fx_rates()
print(f"\nFX rates available for: {sorted(FX_RATES.keys())}")


### 5. Build `prices_db.json` — current state

In [ ]:
def build_prices_db(reference_data, fx_rates):
    """Build the current prices_db.json from reference data."""
    now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    data = {}
    
    for country, vals in reference_data.items():
        iso2, region, currency, gas_loc, die_loc, ref_fx, source_url, eff_date, confidence, cycle = vals
        
        # Use today's FX rate if available, else reference FX
        fx_rate = fx_rates.get(currency, ref_fx)
        fx_source = "frankfurter" if currency in fx_rates and currency not in {
            "LYD","SDG","SSP","ERN","SLL","STN","AOA","CDF","ZWG"} else "fixed"
        
        gas_usd = round(gas_loc / fx_rate, 4) if fx_rate > 0 else 0.0
        die_usd = round(die_loc / fx_rate, 4) if fx_rate > 0 else 0.0
        
        old_source = (country == "Eritrea")  # only Eritrea has 2016 data
        stale = False
        
        data[country] = {
            "iso2": iso2,
            "region": region,
            "currency": currency,
            "gas_loc": round(gas_loc, 4),
            "die_loc": round(die_loc, 4),
            "gas_usd": gas_usd,
            "die_usd": die_usd,
            "fx_rate": round(fx_rate, 4),
            "fx_source": fx_source,
            "fx_date": TODAY.isoformat(),
            "source_url": source_url,
            "source_name": f"{country} — {source_url.split('/')[2]}",
            "effective_date": eff_date,
            "scraped_at": now,
            "status": "ok",
            "stale": stale,
            "old_source": old_source,
            "confidence": confidence,
            "delta_gas_usd": 0.0,
            "delta_die_usd": 0.0,
        }
    
    meta = {
        "last_updated": now,
        "run_date": TODAY.isoformat(),
        "countries_total": len(data),
        "countries_ok": len(data),
        "countries_stale": 0,
        "countries_error": 0,
        "note": "Initial state — built from validated reference data",
    }
    return {"meta": meta, "data": data}


prices_db = build_prices_db(REFERENCE_DATA, FX_RATES)
print(f"✅ prices_db built: {len(prices_db['data'])} countries")
print(f"   Sample: Kenya gas=${prices_db['data']['Kenya']['gas_usd']:.3f}/L")
print(f"   Sample: Nigeria gas={prices_db['data']['Nigeria']['gas_loc']:.0f} NGN")


### 6. Build `history_db.json` — Jan 2026 → today

For the historical period, we use three strategies:
- **Strategy A** — Countries with no price change since Jan 1 2026: one entry per known price change date
- **Strategy B** — BCEAO UEMOA countries: fetch monthly bulletin archives for XOF prices
- **Strategy C** — Countries with documented price changes: interpolate known change dates


In [ ]:
# Known price change history (date → gas_loc, die_loc) per country
# Source: verified from regulators, press archives, Excel corrections
PRICE_HISTORY = {
    "Ethiopia": [
        ("2026-01-07",  101.47, 98.98),   # Ministry of Trade Jan-2026
        ("2026-03-10",  132.18, 139.84),  # Ministry of Trade Mar-2026
    ],
    "Kenya": [
        ("2026-01-15",  182.52, 170.47),  # EPRA Jan-Feb cycle
        ("2026-02-15",  178.28, 166.54),  # EPRA Feb-Mar cycle
        ("2026-03-15",  178.28, 166.54),  # EPRA Mar-Apr cycle (unchanged)
    ],
    "Tanzania": [
        ("2026-01-07",  2877.0, 2767.0),  # EWURA Jan
        ("2026-02-04",  2788.0, 2701.0),  # EWURA Feb
        ("2026-03-04",  2864.0, 2858.0),  # EWURA Mar
    ],
    "South Africa": [
        ("2026-01-07",  21.20,  22.15),   # DMRE Jan
        ("2026-02-04",  20.69,  20.72),   # DMRE Feb
        ("2026-03-04",  19.89,  21.27),   # DMRE Mar
    ],
    "Zambia": [
        ("2026-01-31",  27.88,  24.50),   # ERB Jan
        ("2026-02-28",  26.61,  23.25),   # ERB Feb
    ],
    "Mauritius": [
        ("2026-01-01",  58.95,  58.95),   # STC Jan
        ("2026-02-01",  58.95,  58.95),   # STC Feb (unchanged)
        ("2026-03-24",  58.45,  64.80),   # STC PPC Mar-2026
    ],
    "Rwanda": [
        ("2026-01-05",  2003.0, 1963.0),  # RURA Jan
        ("2026-03-05",  1989.0, 1948.0),  # RURA Mar
    ],
    "Senegal": [
        ("2025-12-06",  920.0,  680.0),   # CRSE decree (last change)
    ],
    "Morocco": [
        # Weekly market — use monthly averages
        ("2026-01-01",  12.80,  11.50),
        ("2026-02-01",  13.10,  11.90),
        ("2026-03-16",  13.82,  12.76),
    ],
    "Nigeria": [
        ("2026-01-01",  950.0,  1200.0),
        ("2026-02-01",  1050.0, 1400.0),
        ("2026-03-18",  1130.0, 1600.0),
    ],
    "Sudan": [
        ("2026-03-29",  630.0,  590.0),   # oilpricez confirmed
    ],
    "South Sudan": [
        ("2026-03-06",  12000.0, 12000.0),# Eye Radio Mar-2026
    ],
}

def build_history_db(reference_data, fx_rates, price_history):
    """
    Build history_db.json:
    - For countries with known history: use documented change dates
    - For stable countries: one entry on START_DATE + one today
    - All entries computed with reference FX
    """
    history = {}
    
    for country, vals in reference_data.items():
        iso2, region, currency, gas_loc, die_loc, ref_fx, source_url, eff_date, confidence, cycle = vals
        fx = fx_rates.get(currency, ref_fx)
        entries = []
        
        if country in price_history:
            # Use documented change dates
            for change_date, g, d in sorted(price_history[country]):
                entries.append({
                    "date": change_date,
                    "gas_loc": g, "die_loc": d,
                    "gas_usd": round(g / fx, 4),
                    "die_usd": round(d / fx, 4),
                    "fx_rate": round(fx, 4),
                    "status": "ok",
                    "source_url": source_url,
                })
        else:
            # Single known value — add it at the effective date
            # If effective_date is before START_DATE, use START_DATE
            eff = eff_date if eff_date >= "2026-01-01" else "2026-01-01"
            entries.append({
                "date": eff,
                "gas_loc": gas_loc, "die_loc": die_loc,
                "gas_usd": round(gas_loc / fx, 4),
                "die_usd": round(die_loc / fx, 4),
                "fx_rate": round(fx, 4),
                "status": "ok",
                "source_url": source_url,
            })
        
        # Add today's entry if not already there
        today_str = TODAY.isoformat()
        if not any(e["date"] == today_str for e in entries):
            entries.append({
                "date": today_str,
                "gas_loc": gas_loc, "die_loc": die_loc,
                "gas_usd": round(gas_loc / fx, 4),
                "die_usd": round(die_loc / fx, 4),
                "fx_rate": round(fx, 4),
                "status": "ok",
                "source_url": source_url,
            })
        
        history[country] = sorted(entries, key=lambda x: x["date"])
    
    return history

history_db = build_history_db(REFERENCE_DATA, FX_RATES, PRICE_HISTORY)
total_entries = sum(len(v) for v in history_db.values())
print(f"✅ history_db built: {len(history_db)} countries, {total_entries} total entries")
print(f"   Kenya history: {len(history_db['Kenya'])} entries")
print(f"   Ethiopia history: {len(history_db['Ethiopia'])} entries")
print(f"   Libya history: {len(history_db['Libya'])} entries (stable)")


### 7. Validation

In [ ]:
def validate_all(prices_db, history_db):
    errors = []
    warnings = []
    
    data = prices_db["data"]
    
    for country, d in data.items():
        # Check all required fields present
        for field in ["iso2","region","currency","gas_loc","die_loc","gas_usd","die_usd","fx_rate","source_url","effective_date","confidence"]:
            if field not in d:
                errors.append(f"{country}: missing field '{field}'")
        
        # Check no null values
        if not d.get("gas_loc") or d["gas_loc"] <= 0:
            errors.append(f"{country}: gas_loc invalid ({d.get('gas_loc')})")
        if not d.get("die_loc") or d["die_loc"] <= 0:
            errors.append(f"{country}: die_loc invalid ({d.get('die_loc')})")
        if not d.get("gas_usd") or d["gas_usd"] <= 0:
            errors.append(f"{country}: gas_usd invalid ({d.get('gas_usd')})")
        
        # Check USD prices are reasonable (0.01 - 10 USD/L)
        if d.get("gas_usd") and not (0.01 <= d["gas_usd"] <= 10):
            warnings.append(f"{country}: gas_usd={d['gas_usd']:.3f} seems unusual")
        
        # Check history exists
        if country not in history_db:
            errors.append(f"{country}: missing from history_db")
        elif not history_db[country]:
            errors.append(f"{country}: empty history")
    
    print(f"Validation: {len(data)} countries")
    if errors:
        print(f"❌ {len(errors)} errors:")
        for e in errors: print(f"   {e}")
    else:
        print("✅ No errors")
    
    if warnings:
        print(f"⚠️  {len(warnings)} warnings:")
        for w in warnings: print(f"   {w}")
    else:
        print("✅ No warnings")
    
    return len(errors) == 0

ok = validate_all(prices_db, history_db)


### 8. Summary table

In [ ]:
print(f"{'Country':<22} {'Cur':<5} {'Gas USD':>8} {'Die USD':>8} {'Conf':<8} {'Eff Date':<12} {'Status'}")
print("-"*80)

REGIONS_ORDER = ["North Africa","West Africa","Central Africa","East Africa","Southern Africa"]
sorted_countries = sorted(prices_db["data"].items(),
    key=lambda x: (REGIONS_ORDER.index(x[1]["region"]) if x[1]["region"] in REGIONS_ORDER else 99, x[0]))

prev_region = None
for country, d in sorted_countries:
    if d["region"] != prev_region:
        print(f"\n── {d['region']} ──")
        prev_region = d["region"]
    old = " [OLD SRC]" if d.get("old_source") else ""
    print(f"  {country:<20} {d['currency']:<5} ${d['gas_usd']:>7.3f} ${d['die_usd']:>7.3f}  {d['confidence']:<8} {d['effective_date']:<12}{old}")

print(f"\n{'='*80}")
print(f"Total: {len(prices_db['data'])} countries")
conf_counts = {}
for d in prices_db['data'].values():
    conf_counts[d['confidence']] = conf_counts.get(d['confidence'], 0) + 1
for k, v in sorted(conf_counts.items()):
    print(f"  {k}: {v}")


### 9. Save `prices_db.json` and `history_db.json`

In [ ]:
import os

def atomic_write(path, data):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

# Save
atomic_write(PRICES_DB,  prices_db)
atomic_write(HISTORY_DB, history_db)

prices_size  = PRICES_DB.stat().st_size / 1024
history_size = HISTORY_DB.stat().st_size / 1024

print(f"✅ prices_db.json  saved → {PRICES_DB}")
print(f"   Size: {prices_size:.1f} KB | {len(prices_db['data'])} countries")
print(f"")
print(f"✅ history_db.json saved → {HISTORY_DB}")
print(f"   Size: {history_size:.1f} KB | {sum(len(v) for v in history_db.values())} total entries")
print(f"")
print("🚀 Ready! You can now:")
print("   1. Run: python run_all_scrapers.py   (first live update)")
print("   2. Run: python generate_dashboard.py (generate HTML)")
print("   3. Push to GitHub → Actions will run daily at 05:00 UTC")


### 10. Optional — Fetch BCEAO monthly bulletins for UEMOA history

This cell fetches the BCEAO monthly bulletins (section 2.3.3) for Jan–Mar 2026
to enrich the history of the 8 UEMOA countries with official XOF prices.
Run only if you want richer historical data. Takes ~2 minutes.


In [ ]:
# Optional — uncomment to run
# BCEAO bulletin URLs (section 2.3.3 contains fuel prices per UEMOA country)
BCEAO_BULLETINS = {
    "2026-01": "https://www.bceao.int/sites/default/files/2026-02/Bulletin%20mensuel%20des%20statistiques%20-%20Janvier%202026.pdf",
    "2026-02": "https://www.bceao.int/sites/default/files/2026-03/Bulletin%20mensuel%20des%20statistiques%20-%20Février%202026.pdf",
}

UEMOA_COUNTRIES = ["Benin","Burkina Faso","Ivory Coast","Guinea-Bissau","Mali","Niger","Senegal","Togo"]

# def fetch_bceao_prices(url):
#     """Fetch and parse a BCEAO bulletin PDF for UEMOA fuel prices."""
#     resp = requests.get(url, timeout=30)
#     # PDF parsing would require pdfplumber — skipped here
#     # The relevant section is 2.3.3 "Evolutions des prix en fin de mois des carburants"
#     return {}
# 
# for month, url in BCEAO_BULLETINS.items():
#     prices = fetch_bceao_prices(url)
#     print(f"BCEAO {month}: {prices}")

print("ℹ️  BCEAO archive fetch is optional.")
print("   The daily scraper will maintain history going forward automatically.")
print("   For Jan-Feb 2026 UEMOA history, check: https://www.bceao.int/fr/publications/bulletins")
